<a href="https://colab.research.google.com/github/aishwaryamaheshb-coder/5L-AI-Lab/blob/main/1WA24CS026_Week7_FOL_Unification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
FAIL = None
def parse(s):
    s = s.replace(" ", "")
    term, rest = _parse(s)
    if rest:
        raise ValueError("Unexpected text: " + rest)
    return term
def _parse(s):
    i = 0
    while i < len(s) and (s[i].isalnum() or s[i] == "_"):
        i += 1
    name, rest = s[:i], s[i:]
    if rest.startswith("("):
        args, rest = [], rest[1:]
        while True:
            arg, rest = _parse(rest)
            args.append(arg)
            if rest.startswith(","):
                rest = rest[1:]
            elif rest.startswith(")"):
                return (name, tuple(args)), rest[1:]
            else:
                raise ValueError("Bad syntax near: " + rest)
    return name, rest
def is_variable(t):
    return isinstance(t, str) and t[0].islower()
def is_compound(t):
    return isinstance(t, tuple)
def to_str(t):
    if is_compound(t):
        return f"{t[0]}({', '.join(to_str(a) for a in t[1])})"
    return t
def unify(x, y, theta):
    if theta is FAIL:
        return FAIL
    if x == y:
        return theta
    if is_variable(x):
        return unify_var(x, y, theta)
    if is_variable(y):
        return unify_var(y, x, theta)
    if is_compound(x) and is_compound(y):
        if x[0] != y[0]:
            print(f"   FAIL: symbols differ ({x[0]} vs {y[0]})")
            return FAIL
        if len(x[1]) != len(y[1]):
            print(f"   FAIL: argument count differs for {x[0]}")
            return FAIL
        for a, b in zip(x[1], y[1]):
            theta = unify(a, b, theta)
            if theta is FAIL:
                return FAIL
        return theta
    print(f"   FAIL: cannot unify {to_str(x)} with {to_str(y)}")
    return FAIL
def unify_var(var, x, theta):
    if var in theta:
        return unify(theta[var], x, theta)
    if is_variable(x) and x in theta:
        return unify(var, theta[x], theta)
    if occurs(var, x, theta):
        print(f"   FAIL: occurs check ({var} occurs in {to_str(x)})")
        return FAIL
    new = dict(theta)
    new[var] = x
    print(f"   {var} / {to_str(x)}")
    return new
def occurs(var, x, theta):
    if var == x:
        return True
    if is_variable(x) and x in theta:
        return occurs(var, theta[x], theta)
    if is_compound(x):
        return any(occurs(var, a, theta) for a in x[1])
    return False
def substitute(t, theta):
    if is_variable(t) and t in theta:
        return substitute(theta[t], theta)
    if is_compound(t):
        return (t[0], tuple(substitute(a, theta) for a in t[1]))
    return t
def run(e1, e2):
    print(f"\nUNIFY( {e1} , {e2} )")
    t1, t2 = parse(e1), parse(e2)
    theta = unify(t1, t2, {})
    if theta is FAIL:
        print("Result: Unification FAILED")
    else:
        mgu = {v: to_str(substitute(v, theta)) for v in theta}
        print("MGU   :", "{" + ", ".join(f"{v}/{t}" for v, t in mgu.items()) + "}")
        print("Unified expression:", to_str(substitute(t1, theta)))
if __name__ == "__main__":
    tests = [
        ("Eats(x, Apple)", "Eats(Riya, y)"),
        ("Knows(John, x)", "Knows(y, Mother(y))"),
        ("P(x, F(y))", "P(A, F(G(z)))"),
        ("Knows(John, x)", "Knows(x, Elisabeth)"),   # fails
        ("P(x, H(y))", "P(A, F(z))"),                # fails: h vs f
        ("P(F(A), G(y))", "P(x, x)"),                # fails
        ("P(x)", "P(F(x))"),                         # fails: occurs check
    ]
    for a, b in tests:
        run(a, b)
    print("\n--- Try your own (leave blank to quit) ---")
    while True:
        a = input("Expression 1: ").strip()
        if not a:
            break
        b = input("Expression 2: ").strip()
        run(a, b)


UNIFY( Eats(x, Apple) , Eats(Riya, y) )
   x / Riya
   y / Apple
MGU   : {x/Riya, y/Apple}
Unified expression: Eats(Riya, Apple)

UNIFY( Knows(John, x) , Knows(y, Mother(y)) )
   y / John
   x / Mother(y)
MGU   : {y/John, x/Mother(John)}
Unified expression: Knows(John, Mother(John))

UNIFY( P(x, F(y)) , P(A, F(G(z))) )
   x / A
   y / G(z)
MGU   : {x/A, y/G(z)}
Unified expression: P(A, F(G(z)))

UNIFY( Knows(John, x) , Knows(x, Elisabeth) )
   x / John
   FAIL: cannot unify John with Elisabeth
Result: Unification FAILED

UNIFY( P(x, H(y)) , P(A, F(z)) )
   x / A
   FAIL: symbols differ (H vs F)
Result: Unification FAILED

UNIFY( P(F(A), G(y)) , P(x, x) )
   x / F(A)
   FAIL: symbols differ (F vs G)
Result: Unification FAILED

UNIFY( P(x) , P(F(x)) )
   FAIL: occurs check (x occurs in F(x))
Result: Unification FAILED

--- Try your own (leave blank to quit) ---
Expression 1: 
